# LeetCode 307 — Range Sum Query - Mutable bằng Python

Link đề: https://leetcode.com/problems/range-sum-query-mutable/ · Độ khó: **Medium** · Chủ đề: Design, Binary Indexed Tree, Segment Tree, Array, Divide and Conquer, Sqrt Decomposition

Skeleton LeetCode:

```python
class NumArray:

    def __init__(self, nums: List[int]):

    def update(self, index: int, val: int) -> None:

    def sumRange(self, left: int, right: int) -> int:


# Your NumArray object will be instantiated and called as such:
# obj = NumArray(nums)
# obj.update(index,val)
# param_2 = obj.sumRange(left,right)
```

Ghi chú: đề, ví dụ và constraints lấy từ đề gốc; mọi đáp án trong notebook đều được kiểm chứng bằng code: ta tự dựng khung test `run(cls, ops, args)` mô phỏng định dạng của LeetCode và đối chiếu với một mô hình độc lập (list thường, cộng `sum(nums[l:r + 1])`) trên chuỗi thao tác ngẫu nhiên có seed.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề **thiết kế cấu trúc dữ liệu**: mảng `nums` ban đầu; hỗ trợ nhiều thao tác **`update(index, val)`** (gán `nums[index] = val`) xen kẽ với **`sumRange(left, right)`** (tổng các phần tử từ `left` đến `right`, tính cả hai đầu).
2. Thấy **cái bẫy cân bằng**: nếu chỉ lưu mảng thì `sumRange` tốn `O(n)`; nếu lưu **tổng tiền tố** (bài 303) thì `sumRange` là `O(1)` nhưng mỗi `update` làm hỏng `O(n)` tổng tiền tố. Với tối đa `3·10⁴` lần gọi, ta cần cân bằng cả hai thao tác.
3. Đi từ **mảng thường (`update` `O(1)`, `sumRange` `O(n)`)** → **tổng tiền tố dựng lại sau mỗi `update`** → **phân rã căn bậc hai (chia khối, `O(√n)` mỗi thao tác)** → **cây Fenwick / Binary Indexed Tree (`O(log n)`, cách tham khảo)** → **cây phân đoạn (segment tree) lặp từ dưới lên (`O(log n)`)**.
4. Hiểu **mẹo bit `x & -x` (lowbit)** của cây Fenwick, cấu trúc cây nhị phân ẩn trong chỉ số `1..n`, và vì sao `update` đi **lên** còn `query` đi **xuống** theo lowbit.
5. Tránh các lỗi kinh điển: dùng chỉ số 0 trong Fenwick (vòng lặp vô hạn), cập nhật bằng `val` thay vì **độ chênh `val − cũ`**, lệch một đơn vị khi lấy `query(right) − query(left − 1)`, và quên dựng lại cấu trúc phụ trợ sau `update`.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`, xử lý nhiều truy vấn thuộc hai loại:

- **Cập nhật** giá trị của một phần tử trong `nums`;
- **Tính tổng** các phần tử của `nums` giữa hai chỉ số `left` và `right` (bao gồm cả hai đầu, `left <= right`).

Cài đặt lớp `NumArray`:

- `NumArray(int[] nums)`: khởi tạo đối tượng với mảng `nums`;
- `void update(int index, int val)`: cập nhật `nums[index] = val`;
- `int sumRange(int left, int right)`: trả về `nums[left] + nums[left + 1] + ... + nums[right]`.

**Ví dụ 1**

```
Input
["NumArray", "sumRange", "update", "sumRange"]
[[[1, 3, 5]], [0, 2], [1, 2], [0, 2]]
Output
[null, 9, null, 8]
```

Giải thích: `numArray = NumArray([1, 3, 5])`; `sumRange(0, 2) = 1 + 3 + 5 = 9`; `update(1, 2)` khiến `nums = [1, 2, 5]`; `sumRange(0, 2) = 1 + 2 + 5 = 8`.

**Ràng buộc:** `1 <= nums.length <= 3 * 10⁴`; `-100 <= nums[i] <= 100`; `0 <= index < nums.length`; `-100 <= val <= 100`; `0 <= left <= right < nums.length`; tối đa `3 * 10⁴` lần gọi `update` và `sumRange`.

In [1]:
from typing import List
import random

ex_ops = ["NumArray", "sumRange", "update", "sumRange"]
ex_args = [[[1, 3, 5]], [0, 2], [1, 2], [0, 2]]
ex_expected = [None, 9, None, 8]


def run(cls, ops, args):
    """Mô phỏng định dạng test của LeetCode: ops[0] là hàm tạo, các lệnh sau gọi phương thức tương ứng."""
    obj = None
    out = []
    for position, (op, arg) in enumerate(zip(ops, args)):
        if position == 0:
            obj = cls(*arg)
            out.append(None)
        else:
            out.append(getattr(obj, op)(*arg))
    return out


print("ops     =", ex_ops)
print("args    =", ex_args)
print("kỳ vọng =", ex_expected)

ops     = ['NumArray', 'sumRange', 'update', 'sumRange']
args    = [[[1, 3, 5]], [0, 2], [1, 2], [0, 2]]
kỳ vọng = [None, 9, None, 8]


## 3. Kiến thức nền

### 3.1. Tổng tiền tố (prefix sum) và vì sao `update` làm nó hỏng

`prefix[i] = nums[0] + ... + nums[i − 1]` (với `prefix[0] = 0`); khi đó `sumRange(l, r) = prefix[r + 1] − prefix[l]` trong `O(1)`. Nhưng nếu đổi `nums[k]` thì mọi `prefix[i]` với `i > k` đều đổi: phải sửa `O(n)` phần tử.

In [2]:
nums = [1, 3, 5, 2, 4]
prefix = [0]
for x in nums:
    prefix.append(prefix[-1] + x)
print(prefix)                                   # [0, 1, 4, 9, 11, 15]
print("sumRange(1, 3) =", prefix[4] - prefix[1])  # 3 + 5 + 2 = 10
nums[1] = 7                                      # sửa một phần tử
print("prefix cũ sau khi sửa nums[1] vẫn là", prefix, "-> đã sai (cần cập nhật các phần tử từ chỉ số 2)")

[0, 1, 4, 9, 11, 15]
sumRange(1, 3) = 10
prefix cũ sau khi sửa nums[1] vẫn là [0, 1, 4, 9, 11, 15] -> đã sai (cần cập nhật các phần tử từ chỉ số 2)


### 3.2. Chia khối (sqrt decomposition)

Chia mảng thành các khối kích thước `b ≈ √n` và lưu **tổng của từng khối**. Truy vấn `[l, r]` = (các phần tử lẻ ở khối đầu) + (tổng các khối nguyên ở giữa) + (các phần tử lẻ ở khối cuối): tối đa `O(b + n/b) = O(√n)`. `update` chỉ sửa một phần tử và một tổng khối: `O(1)`. Chọn `b = ⌈√n⌉` để cân bằng.

In [3]:
from math import isqrt

nums = list(range(1, 11))
n = len(nums)
b = isqrt(n - 1) + 1                            # kích thước khối ≈ căn n (làm tròn lên)
blocks = [sum(nums[i:i + b]) for i in range(0, n, b)]
print("kích thước khối:", b, "| tổng từng khối:", blocks)

kích thước khối: 4 | tổng từng khối: [10, 26, 19]


### 3.3. Toán tử bit `x & -x` (lowbit)

`x & -x` cho ra **bit 1 thấp nhất** của `x` (số nhị phân chỉ giữ lại bit `1` cuối cùng). Ví dụ `12 = 1100₂` có `lowbit = 100₂ = 4`; `6 = 110₂` có `lowbit = 2`. Số này cho biết **độ dài đoạn** mà ô Fenwick `c[x]` phụ trách.

In [4]:
for x in range(1, 17):
    print(f"x={x:2} ({x:05b}): lowbit = {x & -x:2}  -> c[{x}] phụ trách đoạn ({x - (x & -x) + 1}..{x})")

x= 1 (00001): lowbit =  1  -> c[1] phụ trách đoạn (1..1)
x= 2 (00010): lowbit =  2  -> c[2] phụ trách đoạn (1..2)
x= 3 (00011): lowbit =  1  -> c[3] phụ trách đoạn (3..3)
x= 4 (00100): lowbit =  4  -> c[4] phụ trách đoạn (1..4)
x= 5 (00101): lowbit =  1  -> c[5] phụ trách đoạn (5..5)
x= 6 (00110): lowbit =  2  -> c[6] phụ trách đoạn (5..6)
x= 7 (00111): lowbit =  1  -> c[7] phụ trách đoạn (7..7)
x= 8 (01000): lowbit =  8  -> c[8] phụ trách đoạn (1..8)
x= 9 (01001): lowbit =  1  -> c[9] phụ trách đoạn (9..9)
x=10 (01010): lowbit =  2  -> c[10] phụ trách đoạn (9..10)
x=11 (01011): lowbit =  1  -> c[11] phụ trách đoạn (11..11)
x=12 (01100): lowbit =  4  -> c[12] phụ trách đoạn (9..12)
x=13 (01101): lowbit =  1  -> c[13] phụ trách đoạn (13..13)
x=14 (01110): lowbit =  2  -> c[14] phụ trách đoạn (13..14)
x=15 (01111): lowbit =  1  -> c[15] phụ trách đoạn (15..15)
x=16 (10000): lowbit = 16  -> c[16] phụ trách đoạn (1..16)


### 3.4. Cây Fenwick (Binary Indexed Tree) dùng chỉ số từ 1

Mảng `c[1..n]` trong đó `c[x]` lưu **tổng của `lowbit(x)` phần tử** kết thúc ở vị trí `x`: `c[x] = a[x − lowbit(x) + 1] + ... + a[x]` (chỉ số 1-based). Hai thao tác:

- `query(x)` (tổng tiền tố `a[1..x]`): cộng `c[x]`, rồi `x −= lowbit(x)` (nhảy sang đoạn kế tiếp bên trái) cho tới `0`;
- `add(x, delta)` (cộng `delta` vào `a[x]`): cập nhật `c[x]`, rồi `x += lowbit(x)` (nhảy sang ô phụ trách đoạn chứa `x`) cho tới khi vượt `n`.

Mỗi vòng lặp chạy tối đa `⌊log₂ n⌋ + 1` bước. **Bắt buộc dùng chỉ số từ 1**: với `x = 0`, `x & -x = 0` nên `x` không bao giờ thay đổi (vòng lặp vô hạn).

In [5]:
def lowbit(x):
    return x & -x


n = 8
print("query(7) đi qua các ô:", [7, 7 - lowbit(7), 7 - lowbit(7) - lowbit(7 - lowbit(7))])   # 7, 6, 4
x, path = 7, []
while x > 0:
    path.append(x)
    x -= lowbit(x)
print("query(7):", path)                          # [7, 6, 4]
x, path = 3, []
while x <= n:
    path.append(x)
    x += lowbit(x)
print("add(3, ...) cập nhật các ô:", path)         # [3, 4, 8]

query(7) đi qua các ô: [7, 6, 4]
query(7): [7, 6, 4]
add(3, ...) cập nhật các ô: [3, 4, 8]


### 3.5. Cây phân đoạn (segment tree) dạng mảng, dựng từ dưới lên

Dùng mảng `tree` kích thước `2n`: lá ở `tree[n..2n−1]` chứa `nums`, nút trong `tree[i] = tree[2i] + tree[2i + 1]` với `i` từ `n − 1` về `1`. Cách này đúng với **mọi `n`** (không cần `n` là lũy thừa của 2). `update(i, v)`: sửa lá `tree[i + n]` rồi đi lên cha (`i //= 2`) tính lại; `query(l, r)`: dùng hai chỉ số `l`, `r` đi lên, lấy các nút "lẻ" ở biên.

In [6]:
nums = [1, 3, 5]
n = len(nums)
tree = [0] * n + nums
for i in range(n - 1, 0, -1):
    tree[i] = tree[2 * i] + tree[2 * i + 1]
print(tree)                                      # [0, 9, 4, 1, 3, 5]: tree[1] = tổng toàn bộ = 9

[0, 9, 8, 1, 3, 5]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

`nums = [1, 3, 5]`. Ba cách nhìn:

| cách | `sumRange(0, 2)` | `update(1, 2)` | `sumRange(0, 2)` |
|---|---|---|---|
| mảng thường | cộng ba số: 9 | `nums[1] = 2` | cộng lại: `1 + 2 + 5 = 8` |
| tiền tố | `prefix[3] − prefix[0] = 9` | phải sửa `prefix[2] = 3`, `prefix[3] = 8` | `8 − 0 = 8` |
| Fenwick | `query(3) − query(0)` | cộng `delta = 2 − 3 = −1` tại chỉ số `2` | `8` |

Với mảng thường: update rẻ, query đắt. Với tiền tố: query rẻ, update đắt. Ta cần cấu trúc **cả hai đều rẻ**: Fenwick/segment tree, mỗi thao tác đi qua `O(log n)` nút.

**Ý quan trọng của update:** ta cộng **độ chênh** `val − nums[index]` (cần nhớ giá trị cũ) chứ không "gán" vào cấu trúc tổng.

In [7]:
nums = [1, 3, 5]
old = nums[1]
nums[1] = 2
print("độ chênh cần cộng vào các tổng chứa chỉ số 1:", nums[1] - old)       # -1
print("tổng mới:", sum(nums))                                               # 8

độ chênh cần cộng vào các tổng chứa chỉ số 1: -1
tổng mới: 8


## 5. Cách 1 — Brute Force / baseline: lưu mảng, tính tổng khi hỏi

**Ý tưởng.** Lưu `nums`; `update` gán trực tiếp; `sumRange` cộng các phần tử `nums[left..right]` bằng vòng lặp.

In [8]:
class NumArrayNaive:
    def __init__(self, nums: list[int]):
        self.nums = list(nums)

    def update(self, index: int, val: int) -> None:
        self.nums[index] = val                          # O(1)

    def sumRange(self, left: int, right: int) -> int:
        total = 0
        for i in range(left, right + 1):                 # O(right - left + 1)
            total += self.nums[i]
        return total


print(run(NumArrayNaive, ex_ops, ex_args), "| kỳ vọng", ex_expected)

[None, 9, None, 8] | kỳ vọng [None, 9, None, 8]


**Độ phức tạp (Cách 1).** Gọi `n = len(nums)`. Khởi tạo `O(n)`; `update` **`O(1)`**; `sumRange` **`O(n)`**. Với `3·10⁴` thao tác mỗi cái `3·10⁴` phần tử là `9·10⁸`: **quá chậm**. Space `O(n)`.

**Nút thắt.** Mỗi `sumRange` cộng lại từ đầu. Cần cách để `sumRange` không phụ thuộc độ dài đoạn.

## 6. Ý tưởng thuật toán / bước đột phá

**Cần một cấu trúc mà `update` và `sumRange` đều `o(n)`.** Ý tưởng chung: lưu **tổng của một số đoạn được chọn khéo**, sao cho

- một **tổng đoạn bất kỳ** là tổ hợp của `O(log n)` (hoặc `O(√n)`) tổng đã lưu;
- một **thay đổi một phần tử** chỉ ảnh hưởng `O(log n)` (hoặc `O(1)`) tổng đã lưu.

Ba lựa chọn kinh điển:

- **Chia khối `√n`:** đoạn = khối kích thước `√n`;
- **Fenwick:** đoạn = `lowbit(x)` phần tử kết thúc ở `x` (ẩn một cây nhị phân trong chỉ số);
- **Segment tree:** đoạn = nút của cây nhị phân đầy đủ (mỗi nút giữ tổng của khoảng của nó).

Mọi cách dựa trên: **`sumRange(l, r) = prefix(r + 1) − prefix(l)`** (Fenwick) hoặc phân rã `[l, r]` thành các nút (segment tree).

In [9]:
# Minh họa mô hình tổng quát: mỗi lần update chỉ ảnh hưởng O(log n) ô Fenwick; mỗi query dùng O(log n) ô
n = 16
for pos in (1, 5, 9, 16):
    touched, x = [], pos
    while x <= n:
        touched.append(x)
        x += x & -x
    print(f"update tại chỉ số {pos:2}: ảnh hưởng các ô Fenwick {touched}")
for pos in (7, 11, 16):
    used, x = [], pos
    while x > 0:
        used.append(x)
        x -= x & -x
    print(f"query tổng tiền tố {pos:2}: dùng các ô {used}")

update tại chỉ số  1: ảnh hưởng các ô Fenwick [1, 2, 4, 8, 16]
update tại chỉ số  5: ảnh hưởng các ô Fenwick [5, 6, 8, 16]
update tại chỉ số  9: ảnh hưởng các ô Fenwick [9, 10, 12, 16]
update tại chỉ số 16: ảnh hưởng các ô Fenwick [16]
query tổng tiền tố  7: dùng các ô [7, 6, 4]
query tổng tiền tố 11: dùng các ô [11, 10, 8]
query tổng tiền tố 16: dùng các ô [16]


## 7. Các cách tiếp theo

### Cách 2 — Tổng tiền tố, dựng lại sau mỗi `update`

**Giải thích.** Lưu `nums` và `prefix`. `sumRange` = `prefix[right + 1] − prefix[left]` (`O(1)`). `update` sửa `nums[index]` rồi cộng độ chênh vào `prefix[i]` với mọi `i > index` (`O(n)`). Phù hợp khi có rất nhiều truy vấn và rất ít cập nhật.

In [10]:
class NumArrayPrefix:
    def __init__(self, nums: list[int]):
        self.nums = list(nums)
        self.prefix = [0]
        for x in nums:
            self.prefix.append(self.prefix[-1] + x)

    def update(self, index: int, val: int) -> None:
        delta = val - self.nums[index]
        self.nums[index] = val
        for i in range(index + 1, len(self.prefix)):      # mọi tổng tiền tố từ index + 1 trở đi đổi
            self.prefix[i] += delta

    def sumRange(self, left: int, right: int) -> int:
        return self.prefix[right + 1] - self.prefix[left]


print(run(NumArrayPrefix, ex_ops, ex_args), "| kỳ vọng", ex_expected)

[None, 9, None, 8] | kỳ vọng [None, 9, None, 8]


**Độ phức tạp Cách 2:** khởi tạo `O(n)`; `sumRange` **`O(1)`**; `update` **`O(n)`**. Space `O(n)`.

### Cách 3 — Phân rã căn bậc hai (chia khối)

**Giải thích.** Chọn kích thước khối `b = ⌈√n⌉`. Lưu `block_sum[k]` = tổng khối `k` (phần tử `kb .. kb + b − 1`). `update`: sửa `nums[index]` và `block_sum[index // b]` bằng độ chênh. `sumRange(l, r)`: nếu `l` và `r` cùng một khối thì cộng trực tiếp; ngược lại cộng phần lẻ cuối khối của `l`, các khối nguyên ở giữa (qua `block_sum`), và phần lẻ đầu khối của `r`.

In [11]:
class NumArraySqrt:
    def __init__(self, nums: list[int]):
        self.nums = list(nums)
        n = len(nums)
        self.b = isqrt(max(n - 1, 0)) + 1                    # kích thước khối >= 1
        self.block_sum = [0] * ((n + self.b - 1) // self.b)
        for i, x in enumerate(nums):
            self.block_sum[i // self.b] += x

    def update(self, index: int, val: int) -> None:
        self.block_sum[index // self.b] += val - self.nums[index]
        self.nums[index] = val

    def sumRange(self, left: int, right: int) -> int:
        b = self.b
        bl, br = left // b, right // b
        if bl == br:                                          # cùng một khối
            return sum(self.nums[left:right + 1])
        total = sum(self.nums[left:(bl + 1) * b])             # phần lẻ trong khối của left
        total += sum(self.block_sum[bl + 1:br])               # các khối nguyên ở giữa
        total += sum(self.nums[br * b:right + 1])             # phần lẻ trong khối của right
        return total


print(run(NumArraySqrt, ex_ops, ex_args), "| kỳ vọng", ex_expected)

[None, 9, None, 8] | kỳ vọng [None, 9, None, 8]


**Độ phức tạp Cách 3:** khởi tạo `O(n)`; `update` **`O(1)`**; `sumRange` **`O(√n)`** (tối đa `2b` phần tử lẻ + `n/b` khối); Space `O(n)`. Với `n = 3·10⁴`: `√n ≈ 174`, tổng `≈ 10⁷` thao tác: đủ nhanh.

### Cách 4 — Cây Fenwick / Binary Indexed Tree (cách của lời giải tham khảo)

**Giải thích.** `FenwickList(n)` với `c[1..n]`. `update(x, delta)`: `while x ≤ n: c[x] += delta; x += x & −x`. `query(x)`: `s = 0; while x > 0: s += c[x]; x −= x & −x`. `NumArray` giữ thêm `nums` để biết giá trị cũ: `update(index, val)` cộng `val − nums[index]` tại chỉ số `index + 1` (1-based). `sumRange(l, r) = query(r + 1) − query(l)`. Dựng ban đầu trong `O(n)` bằng cách "đẩy" mỗi ô lên cha của nó.

In [12]:
class FenwickList:
    def __init__(self, nums: list[int]):
        self.n = len(nums)
        self.c = [0] + list(nums)                             # c[1..n] ban đầu bằng a[i]
        for i in range(1, self.n + 1):                         # dựng O(n): đẩy tổng ô i lên ô cha
            j = i + (i & -i)
            if j <= self.n:
                self.c[j] += self.c[i]

    def add(self, x: int, delta: int) -> None:                 # x: chỉ số 1-based
        while x <= self.n:
            self.c[x] += delta
            x += x & -x                                         # nhảy lên ô phụ trách đoạn chứa x

    def query(self, x: int) -> int:                            # tổng a[1..x]
        s = 0
        while x > 0:
            s += self.c[x]
            x -= x & -x                                         # nhảy sang đoạn kế bên trái
        return s


class NumArrayBIT:
    def __init__(self, nums: list[int]):
        self.nums = list(nums)
        self.tree = FenwickList(nums)

    def update(self, index: int, val: int) -> None:
        self.tree.add(index + 1, val - self.nums[index])       # cộng độ chênh
        self.nums[index] = val

    def sumRange(self, left: int, right: int) -> int:
        return self.tree.query(right + 1) - self.tree.query(left)


print(run(NumArrayBIT, ex_ops, ex_args), "| kỳ vọng", ex_expected)

[None, 9, None, 8] | kỳ vọng [None, 9, None, 8]


In [13]:
# Code nộp LeetCode cho Cách 4 (viết lại gọn theo lời giải tham khảo)
from typing import List


class BinaryIndexedTree:
    __slots__ = ["n", "c"]

    def __init__(self, n):
        self.n = n
        self.c = [0] * (n + 1)

    def update(self, x: int, delta: int):
        while x <= self.n:
            self.c[x] += delta
            x += x & -x

    def query(self, x: int) -> int:
        s = 0
        while x > 0:
            s += self.c[x]
            x -= x & -x
        return s


class NumArray:
    __slots__ = ["tree"]

    def __init__(self, nums: List[int]):
        self.tree = BinaryIndexedTree(len(nums))
        for i, v in enumerate(nums, 1):
            self.tree.update(i, v)

    def update(self, index: int, val: int) -> None:
        prev = self.sumRange(index, index)
        self.tree.update(index + 1, val - prev)

    def sumRange(self, left: int, right: int) -> int:
        return self.tree.query(right + 1) - self.tree.query(left)

**Độ phức tạp Cách 4:** khởi tạo `O(n log n)` (cách đơn giản cộng từng phần tử) hoặc `O(n)` (cách đẩy lên cha ở trên); `update` và `sumRange` đều **`O(log n)`**; Space `O(n)`. Với `n = 3·10⁴`: tối đa 15 bước mỗi vòng lặp.

### Cách 5 — Cây phân đoạn lặp, dựng từ dưới lên

**Giải thích.** `tree` kích thước `2n`, lá ở `tree[n + i]`. `update(i, v)`: `p = i + n`; `tree[p] = v`; rồi `while p > 1: p //= 2; tree[p] = tree[2p] + tree[2p + 1]`. `sumRange(l, r)`: dùng nửa khoảng `[l, r + 1)`: `l += n; r += n + 1`; trong khi `l < r`: nếu `l` lẻ thì cộng `tree[l]` và `l += 1`; nếu `r` lẻ thì `r −= 1` rồi cộng `tree[r]`; sau đó `l //= 2; r //= 2`.

In [14]:
class NumArraySegmentTree:
    def __init__(self, nums: list[int]):
        self.n = len(nums)
        self.tree = [0] * self.n + list(nums)               # lá ở tree[n .. 2n - 1]
        for i in range(self.n - 1, 0, -1):
            self.tree[i] = self.tree[2 * i] + self.tree[2 * i + 1]

    def update(self, index: int, val: int) -> None:
        p = index + self.n
        self.tree[p] = val
        while p > 1:
            p //= 2                                          # đi lên cha và tính lại tổng
            self.tree[p] = self.tree[2 * p] + self.tree[2 * p + 1]

    def sumRange(self, left: int, right: int) -> int:
        l, r = left + self.n, right + self.n + 1             # nửa khoảng [l, r)
        total = 0
        while l < r:
            if l & 1:
                total += self.tree[l]
                l += 1
            if r & 1:
                r -= 1
                total += self.tree[r]
            l //= 2
            r //= 2
        return total


print(run(NumArraySegmentTree, ex_ops, ex_args), "| kỳ vọng", ex_expected)

[None, 9, None, 8] | kỳ vọng [None, 9, None, 8]


**Độ phức tạp Cách 5:** khởi tạo `O(n)`, `update` và `sumRange` **`O(log n)`**, Space `O(n)` (`2n` ô). Tổng quát hơn Fenwick (dùng được cho `min`, `max`, `gcd`...) nhưng dài hơn.

> Cách 4 (Fenwick) là cách nộp bài. Cách 5 tổng quát hơn. Cách 3 không cần cấu trúc phức tạp. Cách 1 và 2 là các thái cực dùng để đối chiếu.

## 8. TRACE từng bước

Ta theo dõi Fenwick (Cách 4) trên `[1, 3, 5, 2, 4]`: mảng `c` sau khi dựng, rồi từng thao tác `sumRange` và `update`.

In [15]:
def trace_bit(nums, operations):
    bit = FenwickList(nums)
    values = list(nums)
    print("nums =", nums, "| c[1..n] =", bit.c[1:])
    for op in operations:
        if op[0] == "sum":
            _, l, r = op
            path_r, x = [], r + 1
            while x > 0:
                path_r.append(x)
                x -= x & -x
            path_l, x = [], l
            while x > 0:
                path_l.append(x)
                x -= x & -x
            res = bit.query(r + 1) - bit.query(l)
            print(f"  sumRange({l}, {r}): query({r + 1}) qua ô {path_r}, query({l}) qua ô {path_l} -> {res}")
            assert res == sum(values[l:r + 1])
        else:
            _, i, v = op
            delta = v - values[i]
            touched, x = [], i + 1
            while x <= bit.n:
                touched.append(x)
                x += x & -x
            bit.add(i + 1, delta)
            values[i] = v
            print(f"  update({i}, {v}): delta={delta}, cập nhật các ô {touched} -> c = {bit.c[1:]}")


trace_bit([1, 3, 5, 2, 4], [("sum", 0, 2), ("sum", 1, 3), ("update", 2, 1), ("sum", 0, 4), ("sum", 2, 2)])

nums = [1, 3, 5, 2, 4] | c[1..n] = [1, 4, 5, 11, 4]
  sumRange(0, 2): query(3) qua ô [3, 2], query(0) qua ô [] -> 9
  sumRange(1, 3): query(4) qua ô [4], query(1) qua ô [1] -> 10
  update(2, 1): delta=-4, cập nhật các ô [3, 4] -> c = [1, 4, 1, 7, 4]
  sumRange(0, 4): query(5) qua ô [5, 4], query(0) qua ô [] -> 11
  sumRange(2, 2): query(3) qua ô [3, 2], query(2) qua ô [2] -> 1


Bảng tóm tắt (Fenwick cho `[1, 3, 5, 2, 4]`, `n = 5`):

| ô `c[x]` | `lowbit` | đoạn phụ trách | giá trị |
|---|---|---|---|
| `c[1]` | 1 | `a[1..1]` | 1 |
| `c[2]` | 2 | `a[1..2]` | 4 |
| `c[3]` | 1 | `a[3..3]` | 5 |
| `c[4]` | 4 | `a[1..4]` | 11 |
| `c[5]` | 1 | `a[5..5]` | 4 |

Ví dụ `query(4) = c[4] = 11`; `query(5) = c[5] + c[4] = 15`; `query(3) = c[3] + c[2] = 9`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 4 (Fenwick), viết đúng chữ ký của LeetCode. Lớp `NumArray` ghi nhớ giá trị cũ bằng cách tính `sumRange(index, index)` (cách của lời giải tham khảo) nên không cần lưu `nums`.

In [16]:
from typing import List


class BinaryIndexedTree:
    __slots__ = ["n", "c"]

    def __init__(self, n):
        self.n = n
        self.c = [0] * (n + 1)                      # chỉ số từ 1; ô 0 không dùng

    def update(self, x: int, delta: int):
        while x <= self.n:
            self.c[x] += delta
            x += x & -x                              # lên ô phụ trách đoạn lớn hơn chứa x

    def query(self, x: int) -> int:
        s = 0
        while x > 0:
            s += self.c[x]
            x -= x & -x                              # sang đoạn kế bên trái
        return s


class NumArray:
    __slots__ = ["tree"]

    def __init__(self, nums: List[int]):
        self.tree = BinaryIndexedTree(len(nums))
        for i, v in enumerate(nums, 1):
            self.tree.update(i, v)

    def update(self, index: int, val: int) -> None:
        prev = self.sumRange(index, index)            # giá trị cũ
        self.tree.update(index + 1, val - prev)       # cộng độ chênh

    def sumRange(self, left: int, right: int) -> int:
        return self.tree.query(right + 1) - self.tree.query(left)


assert run(NumArray, ex_ops, ex_args) == ex_expected
print("PASS ví dụ đề cho class NumArray")

PASS ví dụ đề cho class NumArray


## 10. Vì sao thuật toán đúng

- **Ý nghĩa của `c[x]` (Fenwick):** `c[x]` giữ tổng `a[x − lowbit(x) + 1 .. x]`. Với `x = 12` (`1100₂`, `lowbit = 4`): `c[12] = a[9..12]`.
- **Truy vấn `query(x)`:** đoạn `[1, x]` phân rã thành các đoạn rời nhau theo các bit `1` của `x`: `x = 7 = 111₂` → `[7, 7]`, `[5, 6]`, `[1, 4]` ứng với các ô `c[7], c[6], c[4]`. Mỗi bước `x −= lowbit(x)` bỏ đi bit `1` thấp nhất, nên đúng các ô cần cộng; có tối đa `⌊log₂ x⌋ + 1` bước.
- **Cập nhật `add(x, δ)`:** ô `c[y]` chứa `a[x]` khi và chỉ khi `y − lowbit(y) + 1 ≤ x ≤ y`. Các `y` như vậy chính là chuỗi `x, x + lowbit(x), ...`: mỗi bước làm bit thấp nhất "tràn" lên bit cao hơn. Cộng `δ` vào đúng các ô đó giữ tính chất của `c`.
- **`sumRange(l, r) = query(r + 1) − query(l)`:** tổng `a[1..r + 1] − a[1..l]` (chỉ số 1-based) = `nums[l] + ... + nums[r]` (0-based).
- **Cập nhật bằng độ chênh:** các ô đang chứa giá trị **cũ** `nums[index]`; cộng `val − cũ` biến nó thành `val`.
- **Cách 5 (segment tree):** nút trong `i` giữ tổng hai con; `update` tính lại tổng dọc đường từ lá tới gốc; `query` lấy các nút "biên" phủ đúng khoảng `[l, r + 1)` mỗi tầng một hoặc hai nút: `O(log n)`.
- **Cách 3:** tổng khối luôn bằng tổng các phần tử khối nhờ `update` cộng độ chênh vào đúng khối.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `Q` là số thao tác.

- **Cách 1 (mảng thường):** `update` `O(1)`, `sumRange` `O(n)`; tổng `O(Q·n)`; Space `O(n)`.
- **Cách 2 (tiền tố dựng lại):** `update` `O(n)`, `sumRange` `O(1)`; tổng `O(Q·n)`; Space `O(n)`.
- **Cách 3 (chia khối):** `update` `O(1)`, `sumRange` `O(√n)`; tổng `O(Q√n)`; Space `O(n)`.
- **Cách 4 (Fenwick):** khởi tạo `O(n)` hoặc `O(n log n)`; `update`, `sumRange` `O(log n)`; tổng `O(Q log n)`; Space `O(n)`.
- **Cách 5 (segment tree):** khởi tạo `O(n)`; `update`, `sumRange` `O(log n)`; Space `O(n)`.
- Với `n, Q = 3·10⁴`: Cách 4, 5 cần cỡ `Q·15 ≈ 4.5·10⁵` bước; Cách 1, 2 cần cỡ `10⁹`.

## 12. Bộ test

Ta gom các lớp vào từ điển `impls`. Mô hình độc lập `Model` là một list thuần với `sum(nums[l:r + 1])` (không dùng tiền tố, khối hay cây). Mỗi test chạy **chuỗi thao tác ngẫu nhiên có seed** trên mọi lớp qua `run`, so sánh kết quả từng bước.

In [17]:
impls = {
    "naive": NumArrayNaive,
    "prefix": NumArrayPrefix,
    "sqrt": NumArraySqrt,
    "bit": NumArrayBIT,
    "segment_tree": NumArraySegmentTree,
    "submit": NumArray,
}
FAST = ("sqrt", "bit", "segment_tree", "submit")


class Model:
    def __init__(self, nums):
        self.nums = list(nums)

    def update(self, index, val):
        self.nums[index] = val

    def sumRange(self, left, right):
        return sum(self.nums[left:right + 1])


def random_case(n, q, lo=-100, hi=100):
    nums = [random.randint(lo, hi) for _ in range(n)]
    ops, args = ["NumArray"], [[nums]]
    for _ in range(q):
        if random.random() < 0.5:
            i = random.randrange(n)
            ops.append("update")
            args.append([i, random.randint(lo, hi)])
        else:
            l = random.randrange(n)
            r = random.randint(l, n - 1)
            ops.append("sumRange")
            args.append([l, r])
    return ops, args


def check_case(ops, args, names=None):
    expected = run(Model, ops, args)
    for name, cls in impls.items():
        if names is not None and name not in names:
            continue
        got = run(cls, ops, [[list(args[0][0])]] + args[1:])
        assert got == expected, (name, ops[:10], got[:10], expected[:10])
    return expected


assert run(Model, ex_ops, ex_args) == ex_expected
for name, cls in impls.items():
    assert run(cls, ex_ops, ex_args) == ex_expected, name
print("PASS ví dụ của đề (mọi lớp và mô hình độc lập)")

PASS ví dụ của đề (mọi lớp và mô hình độc lập)


### 12.1. Các trường hợp biên

In [18]:
def check_ops(nums, operations, label):
    ops = ["NumArray"] + [o[0] for o in operations]
    args = [[nums]] + [list(o[1:]) for o in operations]
    expected = check_case(ops, args)
    print("PASS", label, "->", [e for e in expected if e is not None])


check_ops([5], [("sumRange", 0, 0), ("update", 0, -3), ("sumRange", 0, 0)], "n = 1")
check_ops([1, 2], [("sumRange", 0, 1), ("update", 0, 10), ("sumRange", 0, 1), ("sumRange", 1, 1)], "n = 2")
check_ops([1, 2, 3, 4], [("sumRange", 0, 3), ("update", 3, 0), ("sumRange", 2, 3)], "cập nhật phần tử cuối")
check_ops([1, 2, 3, 4], [("update", 0, 9), ("sumRange", 0, 0), ("sumRange", 0, 3)], "cập nhật phần tử đầu")
check_ops([0, 0, 0], [("sumRange", 0, 2), ("update", 1, 7), ("update", 1, 7), ("sumRange", 0, 2)], "cập nhật cùng giá trị hai lần (độ chênh 0)")
check_ops([-100, 100, -100, 100], [("sumRange", 0, 3), ("update", 0, 100), ("sumRange", 0, 3)], "giá trị biên -100 và 100")
check_ops(list(range(1, 9)), [("sumRange", 0, 7), ("sumRange", 3, 4), ("update", 4, -5), ("sumRange", 3, 4), ("sumRange", 0, 7)], "n = 8 (lũy thừa của 2)")
check_ops(list(range(1, 8)), [("sumRange", 0, 6), ("update", 6, 0), ("sumRange", 5, 6), ("sumRange", 2, 4)], "n = 7 (không phải lũy thừa của 2)")
check_ops(list(range(1, 11)), [("sumRange", 2, 2), ("sumRange", 0, 9), ("sumRange", 8, 9), ("update", 9, 100), ("sumRange", 8, 9)], "n = 10 (biên khối sqrt)")

PASS n = 1 -> [5, -3]
PASS n = 2 -> [3, 12, 2]
PASS cập nhật phần tử cuối -> [10, 3]
PASS cập nhật phần tử đầu -> [9, 18]
PASS cập nhật cùng giá trị hai lần (độ chênh 0) -> [0, 7]
PASS giá trị biên -100 và 100 -> [0, 200]
PASS n = 8 (lũy thừa của 2) -> [36, 9, -1, 26]
PASS n = 7 (không phải lũy thừa của 2) -> [28, 6, 12]
PASS n = 10 (biên khối sqrt) -> [3, 55, 19, 109]


### 12.2. Tính chất và phản ví dụ

In [19]:
random.seed(307)

# Tính chất 1: sumRange(i, i) trả về đúng giá trị phần tử sau mọi update
for name in FAST:
    cls = impls[name]
    for _ in range(50):
        n = random.randint(1, 30)
        nums = [random.randint(-100, 100) for _ in range(n)]
        obj = cls(list(nums))
        for _ in range(60):
            i = random.randrange(n)
            v = random.randint(-100, 100)
            obj.update(i, v)
            nums[i] = v
            assert obj.sumRange(i, i) == v
print("PASS tính chất: sumRange(i, i) = giá trị vừa cập nhật (mọi lớp nhanh)")

# Tính chất 2: sumRange(0, n - 1) bằng tổng toàn mảng; và cộng hai đoạn kề bằng đoạn gộp
for name in FAST:
    cls = impls[name]
    n = 40
    nums = [random.randint(-100, 100) for _ in range(n)]
    obj = cls(list(nums))
    for _ in range(200):
        i = random.randrange(n)
        v = random.randint(-100, 100)
        obj.update(i, v)
        nums[i] = v
        mid = random.randint(0, n - 2)
        assert obj.sumRange(0, n - 1) == sum(nums)
        assert obj.sumRange(0, mid) + obj.sumRange(mid + 1, n - 1) == sum(nums)
print("PASS tính chất: tổng toàn mảng và tính cộng được của hai đoạn kề")

# Phản ví dụ: tiền tố không dựng lại sau update
class SaiTienToKhongCapNhat:
    def __init__(self, nums):
        self.nums = list(nums)
        self.prefix = [0]
        for x in nums:
            self.prefix.append(self.prefix[-1] + x)

    def update(self, index, val):
        self.nums[index] = val                        # sai: quên cập nhật prefix

    def sumRange(self, left, right):
        return self.prefix[right + 1] - self.prefix[left]


print("tiền tố cũ ->", run(SaiTienToKhongCapNhat, ex_ops, ex_args), "| đúng:", ex_expected)
assert run(SaiTienToKhongCapNhat, ex_ops, ex_args) != ex_expected

PASS tính chất: sumRange(i, i) = giá trị vừa cập nhật (mọi lớp nhanh)
PASS tính chất: tổng toàn mảng và tính cộng được của hai đoạn kề
tiền tố cũ -> [None, 9, None, 9] | đúng: [None, 9, None, 8]


### 12.3. Đối chiếu ngẫu nhiên (có seed) với mô hình độc lập

In [20]:
random.seed(2026)
for trial in range(600):
    n = random.randint(1, 40)
    q = random.randint(1, 80)
    ops, args = random_case(n, q)
    check_case(ops, args)

for trial in range(10):                           # n, q lớn theo giới hạn của đề (bỏ qua các lớp chậm)
    n = random.randint(20000, 30000)
    ops, args = random_case(n, 3000)
    check_case(ops, args, names=FAST)

for trial in range(5):                            # n = 3*10^4, 3*10^4 thao tác: chỉ Fenwick và segment tree so với nhau
    n = 30000
    ops, args = random_case(n, 30000)
    got = [run(impls[name], ops, [[list(args[0][0])]] + args[1:]) for name in ("bit", "segment_tree", "submit")]
    assert got[0] == got[1] == got[2]
print("PASS 600 test nhỏ (mọi lớp), 10 test n ~ 2.5*10^4 (lớp nhanh), 5 test 3*10^4 thao tác (Fenwick = segment tree)")

PASS 600 test nhỏ (mọi lớp), 10 test n ~ 2.5*10^4 (lớp nhanh), 5 test 3*10^4 thao tác (Fenwick = segment tree)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng chỉ số 0 trong Fenwick (vòng lặp vô hạn)

In [21]:
class SaiBITChiSo0:
    def __init__(self, nums):
        self.n = len(nums)
        self.c = [0] * self.n                          # sai: mảng 0-based
        for i, v in enumerate(nums):
            self.add(i, v)

    def add(self, x, delta, max_steps=100):
        steps = 0
        while x < self.n and steps < max_steps:        # chặn số bước để minh họa (thực tế sẽ treo)
            self.c[x] += delta
            x += x & -x                                  # với x = 0: x & -x = 0, x không đổi!
            steps += 1
        return steps


b = SaiBITChiSo0([1, 2, 3])
print("số bước khi cập nhật chỉ số 0:", b.add(0, 5), "(đạt giới hạn 100: vòng lặp không bao giờ tiến lên)")

số bước khi cập nhật chỉ số 0: 100 (đạt giới hạn 100: vòng lặp không bao giờ tiến lên)


**Vì sao sai:** `0 & -0 = 0`, nên `x += x & -x` giữ nguyên `x = 0`: vòng lặp vô hạn. Fenwick phải dùng chỉ số **từ 1** (`index + 1`), ô `c[0]` không dùng.

### Lỗi 2: Cập nhật bằng `val` thay vì độ chênh

In [22]:
class SaiBITGanVal:
    def __init__(self, nums):
        self.nums = list(nums)
        self.tree = FenwickList(nums)

    def update(self, index, val):
        self.tree.add(index + 1, val)                   # sai: phải cộng val - nums[index]
        self.nums[index] = val

    def sumRange(self, left, right):
        return self.tree.query(right + 1) - self.tree.query(left)


print("gán val ->", run(SaiBITGanVal, ex_ops, ex_args), "| đúng:", ex_expected)

gán val -> [None, 9, None, 11] | đúng: [None, 9, None, 8]


**Vì sao sai:** cây Fenwick lưu **tổng** nên mỗi ô đang chứa giá trị cũ `nums[index]`. Muốn đổi thành `val` phải cộng **độ chênh** `val − nums[index]`. Cộng `val` làm tổng dư `nums[index]` (ở ví dụ: `2 + 3 = 5` thay vì thay `3` bằng `2`).

### Lỗi 3: Lệch một đơn vị khi tính đoạn `[left, right]`

In [23]:
class SaiBITLechMot(NumArrayBIT):
    def sumRange(self, left, right):
        return self.tree.query(right) - self.tree.query(left)        # sai: thiếu phần tử right và left


print("lệch một ->", run(SaiBITLechMot, ex_ops, ex_args), "| đúng:", ex_expected)

lệch một -> [None, 4, None, 3] | đúng: [None, 9, None, 8]


**Vì sao sai:** `query(x)` là tổng `a[1..x]` theo chỉ số **1-based**. Đoạn `nums[left..right]` (0-based) tương ứng `a[left + 1 .. right + 1]`, nên đáp án là `query(right + 1) − query(left)`. Dùng `query(right) − query(left)` bỏ mất `nums[right]`.

### Lỗi 4: Quên cập nhật cấu trúc phụ trợ (tiền tố / tổng khối)

In [24]:
class SaiKhoiKhongCapNhat(NumArraySqrt):
    def update(self, index, val):
        self.nums[index] = val                             # sai: quên cập nhật block_sum


print("khối không cập nhật ->", run(SaiKhoiKhongCapNhat, ["NumArray", "sumRange", "update", "sumRange"],
                                      [[list(range(1, 11))], [0, 9], [9, 100], [0, 9]]), "| đúng:", run(Model, ["NumArray", "sumRange", "update", "sumRange"], [[list(range(1, 11))], [0, 9], [9, 100], [0, 9]]))

khối không cập nhật -> [None, 55, None, 145] | đúng: [None, 55, None, 145]


**Vì sao sai:** `sumRange` dùng `block_sum` cho các khối nguyên ở giữa; nếu `update` chỉ sửa `nums` thì tổng khối **lạc hậu**. Mọi cấu trúc phụ trợ (tiền tố, tổng khối, cây) phải được cập nhật cùng lúc với dữ liệu gốc.

## 14. Mẹo làm nhanh & nhận dạng pattern

- "Cập nhật một phần tử + truy vấn tổng đoạn": **Fenwick** (ngắn nhất) hoặc **segment tree** (tổng quát); chỉ truy vấn không cập nhật thì dùng **tổng tiền tố** (bài 303).
- Fenwick: chỉ số **từ 1**; `update` đi lên bằng `x += x & -x`, `query` đi xuống bằng `x -= x & -x`; `sumRange(l, r) = query(r + 1) − query(l)`.
- Cập nhật **bằng độ chênh** (cần nhớ giá trị cũ).
- Nếu muốn tránh cấu trúc cây: **chia khối `√n`** dễ viết và dễ mở rộng (cập nhật khoảng, tổng, min...).
- Segment tree lặp dựng từ dưới lên `2n` ô đúng với mọi `n`.
- Khi test bài thiết kế: viết `run(cls, ops, args)` và **mô hình độc lập** (list thuần), kiểm tra bằng chuỗi thao tác ngẫu nhiên có seed.

## 15. Biến thể của bài

1. **Range Sum Query - Immutable (LeetCode 303):** chỉ truy vấn, dùng tổng tiền tố.
2. **Range Sum Query 2D - Mutable (LeetCode 308, Premium):** Fenwick 2 chiều.
3. **Cộng một giá trị vào cả đoạn, truy vấn một điểm:** Fenwick trên mảng hiệu.
4. **Truy vấn `min`/`max` trên đoạn có cập nhật:** segment tree.
5. **Đếm nghịch thế (inversions):** dùng Fenwick trên giá trị.

In [25]:
class FenwickRangeAddPointQuery:
    """Biến thể 3: cộng delta vào mọi phần tử của [l, r]; truy vấn giá trị một phần tử (Fenwick trên mảng hiệu)."""

    def __init__(self, nums):
        self.nums = list(nums)
        self.bit = FenwickList([0] * (len(nums) + 1))

    def range_add(self, left, right, delta):
        self.bit.add(left + 1, delta)
        self.bit.add(right + 2, -delta)

    def point_query(self, i):
        return self.nums[i] + self.bit.query(i + 1)


class SegmentTreeMax:
    """Biến thể 4: max đoạn với cập nhật điểm."""

    def __init__(self, nums):
        self.n = len(nums)
        self.t = [float("-inf")] * self.n + list(nums)
        for i in range(self.n - 1, 0, -1):
            self.t[i] = max(self.t[2 * i], self.t[2 * i + 1])

    def update(self, i, v):
        p = i + self.n
        self.t[p] = v
        while p > 1:
            p //= 2
            self.t[p] = max(self.t[2 * p], self.t[2 * p + 1])

    def query(self, l, r):
        l, r = l + self.n, r + self.n + 1
        best = float("-inf")
        while l < r:
            if l & 1:
                best = max(best, self.t[l])
                l += 1
            if r & 1:
                r -= 1
                best = max(best, self.t[r])
            l //= 2
            r //= 2
        return best


def count_inversions(nums):
    """Biến thể 5: số cặp i < j mà nums[i] > nums[j], bằng Fenwick trên hạng của giá trị (nén tọa độ)."""
    ranks = {v: i + 1 for i, v in enumerate(sorted(set(nums)))}
    bit = FenwickList([0] * len(ranks))
    inv = 0
    for seen, x in enumerate(nums):
        inv += seen - bit.query(ranks[x])              # số phần tử đã gặp lớn hơn x
        bit.add(ranks[x], 1)
    return inv


frp = FenwickRangeAddPointQuery([1, 2, 3, 4, 5])
frp.range_add(1, 3, 10)
assert [frp.point_query(i) for i in range(5)] == [1, 12, 13, 14, 5]
random.seed(15)
for _ in range(100):
    arr = [random.randint(-50, 50) for _ in range(random.randint(1, 20))]
    st = SegmentTreeMax(arr)
    for _ in range(30):
        if random.random() < 0.5:
            i = random.randrange(len(arr))
            v = random.randint(-50, 50)
            arr[i] = v
            st.update(i, v)
        else:
            l = random.randrange(len(arr))
            r = random.randint(l, len(arr) - 1)
            assert st.query(l, r) == max(arr[l:r + 1])
    assert count_inversions(arr) == sum(arr[i] > arr[j] for i in range(len(arr)) for j in range(i + 1, len(arr)))
print("PASS biến thể: cộng đoạn - truy vấn điểm, max đoạn, đếm nghịch thế")

PASS biến thể: cộng đoạn - truy vấn điểm, max đoạn, đếm nghịch thế


## 16. So sánh các cách làm

| Cách | Ý tưởng | `update` | `sumRange` | Space | Khi nào dùng |
|---|---|---|---|---|---|
| 1. Mảng thường | cộng trực tiếp | `O(1)` | `O(n)` | `O(n)` | oracle; ít truy vấn |
| 2. Tiền tố dựng lại | `prefix` sửa `O(n)` mỗi update | `O(n)` | `O(1)` | `O(n)` | ít cập nhật, nhiều truy vấn |
| 3. Chia khối `√n` | tổng khối + phần lẻ | `O(1)` | `O(√n)` | `O(n)` | dễ viết, dễ mở rộng |
| 4. Fenwick (BIT) | `lowbit`, `query(r + 1) − query(l)` | `O(log n)` | `O(log n)` | `O(n)` | **cách nộp bài**, ngắn nhất |
| 5. Segment tree lặp | `2n` ô, tính lại dọc đường lên | `O(log n)` | `O(log n)` | `O(n)` | tổng quát cho `min`, `max`, `gcd` |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `12 & -12`, `7 & -7`, các ô Fenwick `query(6)` đi qua, và `run(NumArrayPrefix, ["NumArray", "sumRange", "update", "sumRange"], [[[4, 5]], [0, 1], [0, 1], [0, 1]])`.

**Bài 2 (tính tay).** Với `nums = [2, 4, 6, 8, 10, 12]`: viết mảng `c[1..6]` của Fenwick, rồi tính `sumRange(1, 4)` bằng hai lần `query`, và mô phỏng `update(2, 0)` (các ô nào thay đổi và đổi bao nhiêu?).

**Bài 3 (điền TODO).** Hoàn thành `class NumArrayTodo` theo Cách 4 (Fenwick).

In [26]:
class NumArrayTodo:
    def __init__(self, nums):
        # TODO 1: self.nums = list(nums); self.n = len(nums); self.c = [0] * (self.n + 1)
        # TODO 2: với mỗi i (1-based) cộng nums[i - 1] vào cây bằng self._add(i, giá_trị)
        pass

    def _add(self, x, delta):
        # TODO 3: while x <= self.n: self.c[x] += delta; x += x & -x
        pass

    def _query(self, x):
        # TODO 4: s = 0; while x > 0: s += self.c[x]; x -= x & -x; trả s
        pass

    def update(self, index, val):
        # TODO 5: _add(index + 1, val - nums[index]) rồi cập nhật nums[index]
        pass

    def sumRange(self, left, right):
        # TODO 6: trả _query(right + 1) - _query(left)
        pass

# KHÔNG tạo đối tượng / gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `class NumArraySegTree2` theo Cách 5 (segment tree `2n` ô dựng từ dưới lên). Kiểm tra bằng `run` với ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `class NumArrayAddRange` hỗ trợ `add_range(l, r, delta)` (cộng `delta` vào mọi phần tử của `[l, r]`) và `sumRange(l, r)` bằng **hai** Fenwick (kỹ thuật cộng đoạn - tổng đoạn). (b) Viết `class NumArray2D` (Fenwick 2 chiều) cho ma trận với `update(r, c, val)` và `sumRegion(r1, c1, r2, c2)`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `x & -x` là bit `1` thấp nhất; `query` trừ `lowbit` cho tới `0`; `update` cộng độ chênh. Bài 2: tính `c[x]` là tổng `lowbit(x)` phần tử kết thúc ở `x`.
- **Mức 2 (vừa):** Bài 1: `4`, `1`, `query(6)` đi qua ô `6` rồi `4`; kết quả `[None, 9, None, 6]` (sau `update(0, 1)` thì `nums = [1, 5]`, tổng `[0, 1]` là `6`). Bài 2: `c = [2, 6, 6, 20, 10, 22]`; `sumRange(1, 4) = query(5) − query(1) = (c[5] + c[4]) − c[1] = 30 − 2 = 28`; `update(2, 0)`: chỉ số 3 (1-based), `delta = −6`: các ô `3, 4` giảm 6 (`c[3] = 0`, `c[4] = 14`). Bài 3: sáu TODO trong comment.
- **Mức 3 (sâu):** Bài 4: `tree = [0] * n + nums`, dựng `i` từ `n − 1` về `1`; `update` đi lên `p //= 2`; `query` nửa khoảng. Bài 5(a): dùng mảng hiệu `d` với hai Fenwick `B1`, `B2`: `prefix(x) = B1.query(x)·x − B2.query(x)`. Bài 5(b): hai vòng lặp lồng nhau `while x <= R` / `while y <= C`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [27]:
# Bài 1
assert (12 & -12) == 4 and (7 & -7) == 1
assert run(NumArrayPrefix, ["NumArray", "sumRange", "update", "sumRange"], [[[4, 5]], [0, 1], [0, 1], [0, 1]]) == [None, 9, None, 6]

# Bài 2
b2 = FenwickList([2, 4, 6, 8, 10, 12])
assert b2.c[1:] == [2, 6, 6, 20, 10, 22]
assert b2.query(5) - b2.query(1) == 28
b2.add(3, -6)
assert b2.c[1:] == [2, 6, 0, 14, 10, 22]

# Bài 3
class NumArrayTodoAns:
    def __init__(self, nums):
        self.nums = list(nums)
        self.n = len(nums)
        self.c = [0] * (self.n + 1)
        for i, v in enumerate(nums, 1):
            self._add(i, v)

    def _add(self, x, delta):
        while x <= self.n:
            self.c[x] += delta
            x += x & -x

    def _query(self, x):
        s = 0
        while x > 0:
            s += self.c[x]
            x -= x & -x
        return s

    def update(self, index, val):
        self._add(index + 1, val - self.nums[index])
        self.nums[index] = val

    def sumRange(self, left, right):
        return self._query(right + 1) - self._query(left)

assert run(NumArrayTodoAns, ex_ops, ex_args) == ex_expected

# Bài 4
class NumArraySegTree2:
    def __init__(self, nums):
        self.n = len(nums)
        self.t = [0] * self.n + list(nums)
        for i in range(self.n - 1, 0, -1):
            self.t[i] = self.t[2 * i] + self.t[2 * i + 1]

    def update(self, index, val):
        p = index + self.n
        self.t[p] = val
        while p > 1:
            p //= 2
            self.t[p] = self.t[2 * p] + self.t[2 * p + 1]

    def sumRange(self, left, right):
        l, r, s = left + self.n, right + self.n + 1, 0
        while l < r:
            if l & 1:
                s += self.t[l]
                l += 1
            if r & 1:
                r -= 1
                s += self.t[r]
            l //= 2
            r //= 2
        return s

assert run(NumArraySegTree2, ex_ops, ex_args) == ex_expected

# Bài 5(a), 5(b)
class NumArrayAddRange:
    def __init__(self, nums):
        self.n = len(nums)
        self.base = [0]
        for x in nums:
            self.base.append(self.base[-1] + x)
        self.b1 = FenwickList([0] * (self.n + 2))
        self.b2 = FenwickList([0] * (self.n + 2))

    def _add(self, i, delta):                       # mảng hiệu tại vị trí i (1-based)
        self.b1.add(i, delta)
        self.b2.add(i, delta * (i - 1))

    def add_range(self, left, right, delta):
        self._add(left + 1, delta)
        self._add(right + 2, -delta)

    def _prefix(self, x):
        return self.base[x] + self.b1.query(x) * x - self.b2.query(x)

    def sumRange(self, left, right):
        return self._prefix(right + 1) - self._prefix(left)

class NumArray2D:
    def __init__(self, matrix):
        self.R, self.C = len(matrix), len(matrix[0])
        self.val = [[0] * self.C for _ in range(self.R)]
        self.c = [[0] * (self.C + 1) for _ in range(self.R + 1)]
        for r in range(self.R):
            for col in range(self.C):
                self.update(r, col, matrix[r][col])

    def update(self, r, col, val):
        delta = val - self.val[r][col]
        self.val[r][col] = val
        x = r + 1
        while x <= self.R:
            y = col + 1
            while y <= self.C:
                self.c[x][y] += delta
                y += y & -y
            x += x & -x

    def _prefix(self, r, col):
        s, x = 0, r
        while x > 0:
            y = col
            while y > 0:
                s += self.c[x][y]
                y -= y & -y
            x -= x & -x
        return s

    def sumRegion(self, r1, c1, r2, c2):
        return self._prefix(r2 + 1, c2 + 1) - self._prefix(r1, c2 + 1) - self._prefix(r2 + 1, c1) + self._prefix(r1, c1)

random.seed(19)
for _ in range(100):
    arr = [random.randint(-9, 9) for _ in range(random.randint(1, 15))]
    ar = NumArrayAddRange(arr)
    for _ in range(30):
        l = random.randrange(len(arr))
        r = random.randint(l, len(arr) - 1)
        if random.random() < 0.5:
            d = random.randint(-5, 5)
            ar.add_range(l, r, d)
            for i in range(l, r + 1):
                arr[i] += d
        else:
            assert ar.sumRange(l, r) == sum(arr[l:r + 1])
    mat = [[random.randint(-9, 9) for _ in range(random.randint(1, 5))] for _ in range(1)]
    R, C = random.randint(1, 5), random.randint(1, 5)
    mat = [[random.randint(-9, 9) for _ in range(C)] for _ in range(R)]
    na = NumArray2D(mat)
    for _ in range(20):
        if random.random() < 0.5:
            r, col, v = random.randrange(R), random.randrange(C), random.randint(-9, 9)
            na.update(r, col, v)
            mat[r][col] = v
        else:
            r1, c1 = random.randrange(R), random.randrange(C)
            r2, c2 = random.randint(r1, R - 1), random.randint(c1, C - 1)
            assert na.sumRegion(r1, c1, r2, c2) == sum(mat[i][j] for i in range(r1, r2 + 1) for j in range(c1, c2 + 1))
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (Fenwick, chỉ số từ 1, cập nhật bằng độ chênh)

```python
class BinaryIndexedTree:
    def __init__(self, n):
        self.n = n
        self.c = [0] * (n + 1)

    def update(self, x: int, delta: int):
        while x <= self.n:
            self.c[x] += delta
            x += x & -x

    def query(self, x: int) -> int:
        s = 0
        while x > 0:
            s += self.c[x]
            x -= x & -x
        return s


class NumArray:
    def __init__(self, nums: List[int]):
        self.tree = BinaryIndexedTree(len(nums))
        for i, v in enumerate(nums, 1):
            self.tree.update(i, v)

    def update(self, index: int, val: int) -> None:
        prev = self.sumRange(index, index)
        self.tree.update(index + 1, val - prev)

    def sumRange(self, left: int, right: int) -> int:
        return self.tree.query(right + 1) - self.tree.query(left)
```

**Checklist**

- [ ] Fenwick dùng chỉ số **từ 1** (`index + 1`), không dùng chỉ số 0 (vòng lặp vô hạn)?
- [ ] `update` cộng **độ chênh** `val − giá_trị_cũ`, không cộng `val`?
- [ ] `sumRange(l, r) = query(r + 1) − query(l)` (không lệch một đơn vị)?
- [ ] Mọi cấu trúc phụ trợ (tiền tố, tổng khối) được cập nhật cùng `update`?
- [ ] Độ phức tạp mỗi thao tác `O(log n)` (hoặc `O(√n)`), không `O(n)`?
- [ ] Đã thử: `n = 1`, cập nhật đầu/cuối, cập nhật cùng giá trị, `n` không phải lũy thừa của 2, giá trị biên `±100`?
- [ ] Đã đối chiếu với mô hình list thuần qua `run` trên chuỗi thao tác ngẫu nhiên có seed?
- [ ] Skeleton đúng `class NumArray` / `update(self, index, val)` / `sumRange(self, left, right)`?